# Synthetic-to-Real Credit Card Fraud Transfer
## Dataset Reconnaissance Notebook

### Objective

This notebook performs a **schema-first audit** of the two mounted datasets:

1. **IBM Credit Card Transactions**
   - `credit_card_transactions-ibm_v2.csv`
   - `User0_credit_card_transactions.csv`
   - `sd254_cards.csv`
   - `sd254_users.csv`

2. **ULB / Worldline Credit Card Fraud Detection**
   - `creditcard.csv`

The notebook is intentionally **non-destructive**:
- no rows are modified
- no preprocessing is applied to the source files
- no resampling is performed
- no model is trained
- the large IBM transaction file is inspected in chunks rather than loaded entirely into RAM

### Research goal

We are investigating whether the IBM transaction environment can support a defensible **synthetic-to-real fraud-transfer study** using the real ULB/Worldline dataset as external validation.

The first task is to determine the exact schemas, entity identifiers, temporal fields, fraud labels, and behavioral information available in the mounted IBM files.

In [ ]:
# ============================================================
# 1. Imports and environment
# ============================================================

import os
import re
import json
import glob
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

from IPython.display import display, Markdown

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 180)

print("Python environment ready.")

## 2. Locate the mounted datasets

The notebook is designed primarily for **Kaggle Notebook** paths (`/kaggle/input`), but it also searches `/mnt/data` and the current working directory.

Because Kaggle may create dataset-folder names that differ from the visible dataset title, the notebook searches by **filename pattern** instead of hard-coding a single folder name.

In [ ]:
# ============================================================
# 2. Discover mounted CSV files
# ============================================================

SEARCH_ROOTS = [
    Path("/kaggle/input"),
    Path("/mnt/data"),
    Path(".")
]

csv_files = []

for root in SEARCH_ROOTS:
    if root.exists():
        try:
            csv_files.extend([p for p in root.rglob("*.csv") if p.is_file()])
        except Exception as e:
            print(f"Could not fully scan {root}: {e}")

# Remove duplicates while preserving order
seen = set()
csv_files_unique = []
for p in csv_files:
    rp = str(p.resolve())
    if rp not in seen:
        seen.add(rp)
        csv_files_unique.append(p)

print(f"CSV files discovered: {len(csv_files_unique)}\n")

for p in csv_files_unique:
    size_mb = p.stat().st_size / (1024**2)
    print(f"{size_mb:10.2f} MB  |  {p}")

In [ ]:
# ============================================================
# 3. Automatically identify the expected files
# ============================================================

def find_by_name(patterns, files):
    matches = []
    for p in files:
        name = p.name.lower()
        if any(pattern.lower() in name for pattern in patterns):
            matches.append(p)
    return matches

ulb_candidates = find_by_name(
    ["creditcard.csv"],
    csv_files_unique
)

ibm_main_candidates = find_by_name(
    ["credit_card_transactions-ibm_v2.csv"],
    csv_files_unique
)

ibm_user0_candidates = find_by_name(
    ["user0_credit_card_transactions.csv"],
    csv_files_unique
)

ibm_cards_candidates = find_by_name(
    ["sd254_cards.csv"],
    csv_files_unique
)

ibm_users_candidates = find_by_name(
    ["sd254_users.csv"],
    csv_files_unique
)

print("ULB candidates:")
for p in ulb_candidates:
    print(" ", p)

print("\nIBM main transaction candidates:")
for p in ibm_main_candidates:
    print(" ", p)

print("\nIBM User0 candidates:")
for p in ibm_user0_candidates:
    print(" ", p)

print("\nIBM cards candidates:")
for p in ibm_cards_candidates:
    print(" ", p)

print("\nIBM users candidates:")
for p in ibm_users_candidates:
    print(" ", p)

if not ulb_candidates:
    raise FileNotFoundError("Could not find creditcard.csv.")

if not ibm_main_candidates:
    raise FileNotFoundError(
        "Could not find credit_card_transactions-ibm_v2.csv. "
        "Run the discovery cell and check the mounted filename."
    )

ULB_PATH = str(ulb_candidates[0])
IBM_MAIN_PATH = str(ibm_main_candidates[0])

IBM_USER0_PATH = str(ibm_user0_candidates[0]) if ibm_user0_candidates else None
IBM_CARDS_PATH = str(ibm_cards_candidates[0]) if ibm_cards_candidates else None
IBM_USERS_PATH = str(ibm_users_candidates[0]) if ibm_users_candidates else None

print("\nSelected files:")
print("ULB       :", ULB_PATH)
print("IBM main  :", IBM_MAIN_PATH)
print("IBM User0 :", IBM_USER0_PATH)
print("IBM cards :", IBM_CARDS_PATH)
print("IBM users :", IBM_USERS_PATH)

## 4. File sizes

This is important because the IBM transaction file may be very large. We will avoid `pd.read_csv()` on the entire file during reconnaissance.

In [ ]:
# ============================================================
# 4. File sizes
# ============================================================

def file_info(path):
    if path is None:
        return None
    p = Path(path)
    size_gb = p.stat().st_size / (1024**3)
    size_mb = p.stat().st_size / (1024**2)
    return {
        "file": p.name,
        "size_MB": round(size_mb, 2),
        "size_GB": round(size_gb, 3),
        "path": str(p)
    }

files_to_report = [
    ULB_PATH,
    IBM_MAIN_PATH,
    IBM_USER0_PATH,
    IBM_CARDS_PATH,
    IBM_USERS_PATH
]

display(pd.DataFrame([file_info(p) for p in files_to_report if p]))

## 5. Load small samples

The IBM main file is read using `nrows` only.

The purpose here is to discover:
- column names
- dtypes
- example values
- likely identifiers
- target/fraud fields
- time/date fields

In [ ]:
# ============================================================
# 5. Sample loading
# ============================================================

SAMPLE_ROWS = 10_000

ibm_sample = pd.read_csv(
    IBM_MAIN_PATH,
    nrows=SAMPLE_ROWS,
    low_memory=False
)

ulb_sample = pd.read_csv(
    ULB_PATH,
    nrows=min(SAMPLE_ROWS, 500_000),
    low_memory=False
)

print("IBM sample shape:", ibm_sample.shape)
print("ULB sample shape:", ulb_sample.shape)

if IBM_USER0_PATH:
    ibm_user0_sample = pd.read_csv(
        IBM_USER0_PATH,
        nrows=10_000,
        low_memory=False
    )
else:
    ibm_user0_sample = None

if IBM_CARDS_PATH:
    ibm_cards_sample = pd.read_csv(
        IBM_CARDS_PATH,
        nrows=10_000,
        low_memory=False
    )
else:
    ibm_cards_sample = None

if IBM_USERS_PATH:
    ibm_users_sample = pd.read_csv(
        IBM_USERS_PATH,
        nrows=10_000,
        low_memory=False
    )
else:
    ibm_users_sample = None

In [ ]:
# ============================================================
# 6. First rows and column lists
# ============================================================

print("=" * 90)
print("IBM MAIN TRANSACTION FILE")
print("=" * 90)
display(ibm_sample.head())

print("\nIBM columns:")
for i, col in enumerate(ibm_sample.columns):
    print(f"{i:3d}: {col}")

print("\n" + "=" * 90)
print("ULB CREDITCARD FILE")
print("=" * 90)
display(ulb_sample.head())

print("\nULB columns:")
for i, col in enumerate(ulb_sample.columns):
    print(f"{i:3d}: {col}")

## 7. Inspect all IBM auxiliary files

The IBM dataset shown in the mounted environment contains four useful components:

- main transaction table
- User0 transaction sample
- card table
- user table

We need to understand their relationships before deciding whether the main transaction table can be used alone or whether the card/user tables should be joined.

In [ ]:
# ============================================================
# 7. Auxiliary IBM schemas
# ============================================================

def show_schema(df, name):
    if df is None:
        print(f"{name}: NOT FOUND")
        return

    print("=" * 90)
    print(name)
    print("=" * 90)
    print("Shape:", df.shape)
    print("\nColumns:")
    for i, col in enumerate(df.columns):
        print(f"{i:3d}: {col}")

    print("\nDtypes:")
    display(df.dtypes.to_frame("dtype"))

    print("\nFirst 5 rows:")
    display(df.head())


show_schema(ibm_user0_sample, "IBM User0 Transactions")
show_schema(ibm_cards_sample, "IBM Cards")
show_schema(ibm_users_sample, "IBM Users")

## 8. Data types and missing values

Missing values matter because a cross-dataset representation should not depend on fields that are sparsely populated.

In [ ]:
# ============================================================
# 8. Missing-value reports
# ============================================================

def missing_report(df, name):
    report = pd.DataFrame({
        "column": df.columns,
        "dtype": [str(df[c].dtype) for c in df.columns],
        "missing_count": [int(df[c].isna().sum()) for c in df.columns],
        "missing_pct": [float(df[c].isna().mean() * 100) for c in df.columns],
        "unique_values": [int(df[c].nunique(dropna=True)) for c in df.columns]
    })

    report = report.sort_values(
        ["missing_pct", "unique_values"],
        ascending=[False, False]
    )

    print(f"\n{name}")
    display(report)

    return report


ibm_missing = missing_report(
    ibm_sample,
    "IBM Main Transaction Missingness"
)

ulb_missing = missing_report(
    ulb_sample,
    "ULB Missingness"
)

In [ ]:
# ============================================================
# 9. Numerical summaries
# ============================================================

def numerical_summary(df, name):
    num_cols = df.select_dtypes(include=np.number).columns.tolist()

    print(f"\n{name} numerical columns:")
    print(num_cols)

    if num_cols:
        summary = df[num_cols].describe(
            percentiles=[
                0.01, 0.05, 0.25, 0.50,
                0.75, 0.95, 0.99
            ]
        ).T

        display(summary)
        return summary

    print("No numerical columns found.")
    return pd.DataFrame()


ibm_numeric_summary = numerical_summary(
    ibm_sample,
    "IBM Main"
)

ulb_numeric_summary = numerical_summary(
    ulb_sample,
    "ULB"
)

In [ ]:
# ============================================================
# 10. Categorical/object columns
# ============================================================

def categorical_summary(df, name, max_display=20):
    cat_cols = df.select_dtypes(
        include=["object", "category"]
    ).columns.tolist()

    print(f"\n{name} categorical/object columns:")
    print(cat_cols)

    for col in cat_cols:
        print(f"\n--- {col} ---")
        print("Unique:", df[col].nunique(dropna=True))

        display(
            df[col]
            .value_counts(dropna=False)
            .head(max_display)
            .to_frame("count")
        )


categorical_summary(
    ibm_sample,
    "IBM Main"
)

categorical_summary(
    ulb_sample,
    "ULB"
)

## 11. Search for target/fraud columns

We should not assume that the IBM fraud target has a particular name.

In [ ]:
# ============================================================
# 11. Target candidates
# ============================================================

TARGET_KEYWORDS = [
    "fraud",
    "fraudulent",
    "is_fraud",
    "fraud_flag",
    "class",
    "label",
    "target",
    "anomaly"
]

def target_candidates(df):
    results = []

    for col in df.columns:
        low = col.lower().replace(" ", "_")

        if any(k in low for k in TARGET_KEYWORDS):
            results.append(col)

    return results


print("IBM target candidates:")
print(target_candidates(ibm_sample))

print("\nULB target candidates:")
print(target_candidates(ulb_sample))

In [ ]:
# ============================================================
# 12. ULB target distribution
# ============================================================

ULB_TARGET = "Class"

print("ULB target distribution:")
display(
    ulb_sample[ULB_TARGET]
    .value_counts(dropna=False)
    .to_frame("count")
)

print("\nULB target percentage:")
display(
    (
        ulb_sample[ULB_TARGET]
        .value_counts(normalize=True, dropna=False) * 100
    ).to_frame("percentage")
)

### IBM fraud target

The following cell automatically selects a likely IBM target if exactly one candidate is found.

If multiple candidates appear, the notebook stops and asks you to inspect them rather than silently choosing the wrong field.

In [ ]:
# ============================================================
# 13. Automatically identify IBM target
# ============================================================

ibm_target_candidates = target_candidates(ibm_sample)

print("Candidates:", ibm_target_candidates)

if len(ibm_target_candidates) == 1:
    IBM_TARGET = ibm_target_candidates[0]
    print("Automatically selected IBM target:", IBM_TARGET)
elif len(ibm_target_candidates) == 0:
    IBM_TARGET = None
    print(
        "No obvious fraud target was detected. "
        "We will inspect the schema manually."
    )
else:
    IBM_TARGET = None
    print(
        "Multiple candidates detected. "
        "Do NOT select one automatically."
    )

if IBM_TARGET is not None:
    print("\nIBM target distribution:")
    display(
        ibm_sample[IBM_TARGET]
        .value_counts(dropna=False)
        .to_frame("count")
    )

    print("\nIBM target percentage:")
    display(
        (
            ibm_sample[IBM_TARGET]
            .value_counts(normalize=True, dropna=False) * 100
        ).to_frame("percentage")
    )

## 14. Entity/identifier discovery

This is one of the most important parts of the reconnaissance.

For the IBM dataset, we want to know whether transactions can be grouped by:
- customer/user
- card
- account
- merchant
- transaction

Repeated customer/card histories would allow us to construct behavioral features such as:
- transaction velocity
- amount deviation
- merchant novelty
- time-of-day deviation
- recent transaction frequency

These may become the foundation of the synthetic-to-real transfer representation.

In [ ]:
# ============================================================
# 14. Identifier and entity candidates
# ============================================================

ENTITY_KEYWORDS = [
    "id",
    "user",
    "customer",
    "client",
    "card",
    "account",
    "merchant",
    "transaction"
]

def entity_candidates(df):
    rows = []

    for col in df.columns:
        low = col.lower()

        if any(k in low for k in ENTITY_KEYWORDS):
            rows.append({
                "column": col,
                "dtype": str(df[col].dtype),
                "unique": int(df[col].nunique(dropna=True)),
                "unique_ratio": float(
                    df[col].nunique(dropna=True) / len(df)
                )
            })

    return pd.DataFrame(rows).sort_values(
        "unique",
        ascending=False
    )


print("IBM entity candidates:")
ibm_entities = entity_candidates(ibm_sample)
display(ibm_entities)

print("\nULB entity candidates:")
ulb_entities = entity_candidates(ulb_sample)
display(ulb_entities)

## 15. IBM time/date field discovery

We need to identify whether IBM has:
- a transaction date
- a transaction timestamp
- an elapsed-time field
- month/year fields

Do not assume that a field called `Time` has the same meaning as ULB `Time`.

In [ ]:
# ============================================================
# 15. Time/date candidates
# ============================================================

TIME_KEYWORDS = [
    "time",
    "date",
    "datetime",
    "timestamp",
    "year",
    "month",
    "day",
    "hour"
]

def time_candidates(df):
    rows = []

    for col in df.columns:
        low = col.lower()

        if any(k in low for k in TIME_KEYWORDS):
            rows.append({
                "column": col,
                "dtype": str(df[col].dtype),
                "sample_values": df[col].dropna().astype(str).head(5).tolist()
            })

    return pd.DataFrame(rows)


print("IBM time/date candidates:")
display(time_candidates(ibm_sample))

print("\nULB time/date fields:")
display(time_candidates(ulb_sample))

In [ ]:
# ============================================================
# 16. ULB time behavior
# ============================================================

print("ULB Time statistics:")
display(ulb_sample["Time"].describe())

print("\nULB Time range:")
print("min:", ulb_sample["Time"].min())
print("max:", ulb_sample["Time"].max())

print(
    "\nULB Time is an elapsed-time field measured in seconds "
    "from the beginning of the recorded period."
)

## 17. IBM cardinality analysis

High-cardinality fields can reveal the structure of the synthetic environment and help us identify whether user/card/merchant histories are available.

In [ ]:
# ============================================================
# 17. IBM cardinality
# ============================================================

def cardinality_report(df):
    rows = []

    for col in df.columns:
        rows.append({
            "column": col,
            "dtype": str(df[col].dtype),
            "unique": int(df[col].nunique(dropna=True)),
            "unique_ratio": float(
                df[col].nunique(dropna=True) / len(df)
            )
        })

    return pd.DataFrame(rows).sort_values(
        "unique",
        ascending=False
    )


ibm_cardinality = cardinality_report(ibm_sample)
display(ibm_cardinality)

## 18. Inspect the ULB fraud/legitimate amount distributions

`Amount` is one of the few ULB variables that retains direct financial meaning.

In [ ]:
# ============================================================
# 18. ULB Amount by class
# ============================================================

for cls, label in [(0, "Legitimate"), (1, "Fraud")]:
    subset = ulb_sample.loc[
        ulb_sample[ULB_TARGET] == cls,
        "Amount"
    ]

    print(f"\nULB {label} transactions:")
    display(
        subset.describe(
            percentiles=[
                0.01, 0.05, 0.25, 0.50,
                0.75, 0.95, 0.99
            ]
        ).to_frame("Amount")
    )

In [ ]:
# ============================================================
# 19. ULB numerical correlation with fraud
# ============================================================

ulb_numeric = ulb_sample.select_dtypes(
    include=np.number
)

if ULB_TARGET in ulb_numeric.columns:
    ulb_corr = (
        ulb_numeric
        .corr(numeric_only=True)[ULB_TARGET]
        .sort_values()
    )

    display(ulb_corr.to_frame("correlation_with_Class"))
else:
    print("ULB target is not numeric.")

## 20. IBM numerical correlation with fraud

This is only exploratory.

We will **not** infer causal relationships from correlation.

In [ ]:
# ============================================================
# 20. IBM numerical correlation with fraud
# ============================================================

if IBM_TARGET is not None:

    ibm_numeric = ibm_sample.select_dtypes(
        include=np.number
    )

    if IBM_TARGET in ibm_numeric.columns:

        ibm_corr = (
            ibm_numeric
            .corr(numeric_only=True)[IBM_TARGET]
            .sort_values()
        )

        display(
            ibm_corr.to_frame(
                "correlation_with_IBM_target"
            )
        )

    else:
        print(
            f"{IBM_TARGET} is not a numerical column."
        )

else:
    print(
        "IBM_TARGET has not been identified yet."
    )

## 21. Duplicate analysis

We need to know whether duplicate transaction records exist in either dataset.

For the IBM 20M file, this is performed on a sample only at this stage.

In [ ]:
# ============================================================
# 21. Duplicate analysis
# ============================================================

print("IBM sample duplicate rows:")
print(
    ibm_sample.duplicated().sum(),
    f"({ibm_sample.duplicated().mean()*100:.4f}%)"
)

print("\nULB duplicate rows:")
print(
    ulb_sample.duplicated().sum(),
    f"({ulb_sample.duplicated().mean()*100:.4f}%)"
)

## 22. Inspect likely identifier relationships among IBM auxiliary tables

If the card and user tables contain foreign keys that correspond to the transaction table, this cell will help identify possible joins.

We intentionally do not perform the joins automatically.

In [ ]:
# ============================================================
# 22. Compare likely key names
# ============================================================

if ibm_cards_sample is not None:
    print("IBM transaction columns:")
    print(list(ibm_sample.columns))

    print("\nIBM cards columns:")
    print(list(ibm_cards_sample.columns))

if ibm_users_sample is not None:
    print("\nIBM users columns:")
    print(list(ibm_users_sample.columns))

In [ ]:
# ============================================================
# 23. Compare common column names
# ============================================================

def common_columns(df1, df2):
    a = {c.lower(): c for c in df1.columns}
    b = {c.lower(): c for c in df2.columns}

    common = sorted(set(a) & set(b))

    return pd.DataFrame({
        "normalized_name": common,
        "df1_column": [a[x] for x in common],
        "df2_column": [b[x] for x in common]
    })


if ibm_cards_sample is not None:
    print("Transaction ↔ Cards common columns:")
    display(
        common_columns(
            ibm_sample,
            ibm_cards_sample
        )
    )

if ibm_users_sample is not None:
    print("Transaction ↔ Users common columns:")
    display(
        common_columns(
            ibm_sample,
            ibm_users_sample
        )
    )

if ibm_cards_sample is not None and ibm_users_sample is not None:
    print("Cards ↔ Users common columns:")
    display(
        common_columns(
            ibm_cards_sample,
            ibm_users_sample
        )
    )

## 24. Large-file row count

This cell counts rows in the IBM main transaction CSV without loading the entire dataset into memory.

It can take some time for a multi-gigabyte file.

In [ ]:
# ============================================================
# 24. Exact row counts
# ============================================================

def count_csv_rows(path):
    count = 0
    with open(path, "rb") as f:
        for _ in f:
            count += 1
    return max(count - 1, 0)


print("Counting IBM main transaction rows...")
IBM_ROWS = count_csv_rows(IBM_MAIN_PATH)

print(f"IBM main transaction rows: {IBM_ROWS:,}")

print("\nCounting ULB rows...")
ULB_ROWS = count_csv_rows(ULB_PATH)

print(f"ULB rows: {ULB_ROWS:,}")

## 25. Chunk-based IBM target distribution

If the IBM file contains a fraud label, we need its distribution over the **full dataset**, not only the 10,000-row sample.

This cell processes the file in chunks.

In [ ]:
# ============================================================
# 25. Full IBM target distribution using chunks
# ============================================================

CHUNK_SIZE = 500_000

if IBM_TARGET is not None:

    target_counts = {}

    for chunk in pd.read_csv(
        IBM_MAIN_PATH,
        usecols=[IBM_TARGET],
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):
        counts = chunk[IBM_TARGET].value_counts(dropna=False)

        for key, value in counts.items():
            target_counts[key] = (
                target_counts.get(key, 0) + int(value)
            )

    target_df = (
        pd.Series(target_counts, name="count")
        .to_frame()
    )

    target_df["percentage"] = (
        target_df["count"] /
        target_df["count"].sum() * 100
    )

    display(target_df)

else:
    print(
        "IBM_TARGET is not identified. "
        "Skipping full target scan."
    )

## 26. Full-file numeric ranges for the IBM main dataset

This uses chunks to calculate approximate global statistics without loading 20M+ rows into memory.

We first identify numerical columns from the sample.

In [ ]:
# ============================================================
# 26. Chunk-based global numeric summary
# ============================================================

IBM_NUMERIC_COLUMNS = ibm_sample.select_dtypes(
    include=np.number
).columns.tolist()

print("IBM numerical columns:")
print(IBM_NUMERIC_COLUMNS)

# We calculate exact count/min/max and approximate moments from chunks.
global_stats = {}

for col in IBM_NUMERIC_COLUMNS:

    if col == IBM_TARGET:
        continue

    global_stats[col] = {
        "count": 0,
        "min": np.inf,
        "max": -np.inf,
        "sum": 0.0,
        "sum_sq": 0.0
    }


usecols = [
    c for c in IBM_NUMERIC_COLUMNS
    if c != IBM_TARGET
]

if usecols:
    for chunk in pd.read_csv(
        IBM_MAIN_PATH,
        usecols=usecols,
        chunksize=CHUNK_SIZE,
        low_memory=False
    ):
        for col in usecols:
            x = pd.to_numeric(
                chunk[col],
                errors="coerce"
            ).dropna()

            if len(x) == 0:
                continue

            arr = x.to_numpy(dtype=np.float64)

            global_stats[col]["count"] += len(arr)
            global_stats[col]["min"] = min(
                global_stats[col]["min"],
                float(arr.min())
            )
            global_stats[col]["max"] = max(
                global_stats[col]["max"],
                float(arr.max())
            )
            global_stats[col]["sum"] += float(arr.sum())
            global_stats[col]["sum_sq"] += float(
                np.square(arr).sum()
            )

    rows = []

    for col, s in global_stats.items():

        n = s["count"]

        if n > 0:
            mean = s["sum"] / n
            variance = max(
                s["sum_sq"] / n - mean**2,
                0
            )
            std = np.sqrt(variance)
        else:
            mean = np.nan
            std = np.nan

        rows.append({
            "column": col,
            "count": n,
            "mean": mean,
            "std": std,
            "min": s["min"],
            "max": s["max"]
        })

    global_numeric_summary = pd.DataFrame(rows)

    display(global_numeric_summary)

else:
    print("No IBM numerical columns available.")

## 27. Check whether IBM has repeated users/cards/merchants

If an entity identifier exists, this chunked scan can calculate the number of unique entities and transaction frequency statistics.

The notebook tries likely identifier columns automatically.

In [ ]:
# ============================================================
# 27. Candidate repeated entities in IBM
# ============================================================

possible_entity_cols = []

for col in ibm_sample.columns:
    low = col.lower()

    if any(
        k in low
        for k in [
            "user",
            "customer",
            "client",
            "card",
            "account",
            "merchant"
        ]
    ):
        possible_entity_cols.append(col)

print("Possible entity columns:")
print(possible_entity_cols)

# Exact unique counts can require a large memory footprint.
# Therefore we estimate frequency structure from the first sample.
for col in possible_entity_cols:

    counts = ibm_sample[col].value_counts(
        dropna=False
    )

    print(f"\n--- {col} ---")
    print("Sample unique entities:", len(counts))
    print(
        "Sample median transactions/entity:",
        counts.median()
    )
    print(
        "Sample mean transactions/entity:",
        counts.mean()
    )
    print(
        "Sample maximum transactions/entity:",
        counts.max()
    )

## 28. Build a preliminary semantic-feature checklist

This does **not** create features yet.

It simply checks whether the raw datasets appear to contain information from which common behavioral concepts might be derived.

Potential shared concepts:

- transaction amount
- transaction time
- transaction frequency / velocity
- amount deviation from historical behavior
- temporal irregularity
- merchant novelty
- card/customer activity
- transaction count in a recent window

The final mapping must be decided after reviewing the actual IBM schema.

In [ ]:
# ============================================================
# 28. Preliminary semantic feature availability
# ============================================================

def find_matching_columns(columns, keywords):
    matches = []

    for col in columns:
        low = col.lower()

        if any(k in low for k in keywords):
            matches.append(col)

    return matches


semantic_groups = {
    "amount": ["amount", "price", "value", "cost"],
    "time": ["time", "date", "timestamp", "datetime", "hour"],
    "user": ["user", "customer", "client", "person"],
    "card": ["card", "account"],
    "merchant": ["merchant", "store", "vendor"],
    "location": ["location", "city", "state", "country", "zip", "lat", "lon"],
    "category": ["category", "mcc", "merchant_category", "type"],
    "transaction_id": ["transaction_id", "trans_id", "txn_id"]
}

rows = []

for group, keywords in semantic_groups.items():

    rows.append({
        "semantic_group": group,
        "IBM_matches": find_matching_columns(
            ibm_sample.columns,
            keywords
        ),
        "ULB_matches": find_matching_columns(
            ulb_sample.columns,
            keywords
        )
    })

semantic_report = pd.DataFrame(rows)

display(semantic_report)

## 29. ULB PCA-feature reminder

The ULB variables `V1`–`V28` are anonymized/PCA-transformed features.

For the eventual paper:

**Do not assign semantic names to individual PCA components.**

We can use them statistically, but if we propose a cross-dataset semantic alignment method, we should only map concepts that have legitimate support in both datasets.

In [ ]:
# ============================================================
# 29. Confirm ULB feature structure
# ============================================================

ulb_pca_cols = [
    c for c in ulb_sample.columns
    if re.fullmatch(r"V\d+", str(c))
]

print("ULB PCA-style columns:")
print(ulb_pca_cols)

print("\nCount:", len(ulb_pca_cols))

print("\nOther ULB columns:")
print([
    c for c in ulb_sample.columns
    if c not in ulb_pca_cols
])

## 30. Save the complete reconnaissance report

The JSON file stores schema-level information so the notebook can be reproduced without sending the datasets themselves.

In [ ]:
# ============================================================
# 30. Save reconnaissance metadata
# ============================================================

recon = {
    "paths": {
        "ULB": ULB_PATH,
        "IBM_MAIN": IBM_MAIN_PATH,
        "IBM_USER0": IBM_USER0_PATH,
        "IBM_CARDS": IBM_CARDS_PATH,
        "IBM_USERS": IBM_USERS_PATH
    },

    "row_counts": {
        "IBM_MAIN": int(IBM_ROWS),
        "ULB": int(ULB_ROWS)
    },

    "IBM_columns": list(ibm_sample.columns),
    "ULB_columns": list(ulb_sample.columns),

    "IBM_dtypes": {
        c: str(ibm_sample[c].dtype)
        for c in ibm_sample.columns
    },

    "ULB_dtypes": {
        c: str(ulb_sample[c].dtype)
        for c in ulb_sample.columns
    },

    "IBM_target_candidates": target_candidates(ibm_sample),
    "ULB_target_candidates": target_candidates(ulb_sample),

    "IBM_entity_candidates": (
        ibm_entities.to_dict(orient="records")
        if len(ibm_entities)
        else []
    ),

    "ULB_entity_candidates": (
        ulb_entities.to_dict(orient="records")
        if len(ulb_entities)
        else []
    ),

    "IBM_time_candidates": (
        time_candidates(ibm_sample).to_dict(
            orient="records"
        )
    ),

    "ULB_time_candidates": (
        time_candidates(ulb_sample).to_dict(
            orient="records"
        )
    )
}

with open(
    "credit_card_dataset_reconnaissance.json",
    "w"
) as f:
    json.dump(
        recon,
        f,
        indent=2,
        default=str
    )

print(
    "Saved: credit_card_dataset_reconnaissance.json"
)

# 31. What to send back

After running the notebook, send the outputs of these sections:

### Most important
1. **IBM MAIN TRANSACTION FILE — first rows and columns**
2. **IBM auxiliary schemas**
3. **IBM target candidates/distribution**
4. **IBM entity candidates**
5. **IBM time/date candidates**
6. **IBM cardinality report**
7. **IBM numerical summary**
8. **ULB columns and first rows**
9. **ULB target distribution**
10. **Semantic feature availability table**

### Especially useful

If the notebook produces:

`credit_card_dataset_reconnaissance.json`

you can also upload that small JSON file.

### Do not train anything yet.

The next stage will be decided from the actual schema. We need to verify that the proposed synthetic-to-real transfer representation is scientifically defensible before implementing the model.